# Loan Propensity Model — Exploratory Analysis
End-to-end walk-through: data exploration → feature engineering → XGBoost training → evaluation.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../app/db/crm.db')
df = pd.read_sql('SELECT * FROM customers', engine)
print(f'Dataset: {len(df)} customers, {df.columns.tolist()}')

## 1. Data Overview

In [ ]:
df.describe(include='all').T[['count','mean','std','min','max']].round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Customer Distribution', fontsize=14)

df['monthly_income'].hist(bins=30, ax=axes[0,0], color='steelblue', edgecolor='white')
axes[0,0].set_title('Monthly Income (₹)')

df['account_balance'].hist(bins=30, ax=axes[0,1], color='seagreen', edgecolor='white')
axes[0,1].set_title('Account Balance (₹)')

df['credit_score'].hist(bins=30, ax=axes[0,2], color='coral', edgecolor='white')
axes[0,2].set_title('Credit Score')

df['age'].hist(bins=20, ax=axes[1,0], color='mediumpurple', edgecolor='white')
axes[1,0].set_title('Age')

df['city'].value_counts().plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Customers by City')
axes[1,1].tick_params(axis='x', rotation=45)

df['occupation'].value_counts().head(8).plot(kind='barh', ax=axes[1,2], color='seagreen')
axes[1,2].set_title('Top Occupations')

plt.tight_layout()
plt.show()

## 2. Feature Engineering & Label Creation

In [ ]:
import numpy as np
np.random.seed(42)

EDU_MAP = {'Undergraduate': 0, 'Graduate': 1, 'Post Graduate': 2, 'Professional': 3, 'Doctorate': 4}
df['education_encoded'] = df['education'].map(EDU_MAP)

def synthetic_label(row):
    if row['has_personal_loan']:
        return 0
    score = 0
    score += 4 if row['credit_score'] >= 780 else (3 if row['credit_score'] >= 750 else (1 if row['credit_score'] >= 720 else 0))
    score += 4 if row['monthly_income'] > 120000 else (2 if row['monthly_income'] > 80000 else (1 if row['monthly_income'] > 50000 else 0))
    score += 2 if row['has_salary_account'] else 0
    score += 2 if row['months_as_customer'] > 48 else (1 if row['months_as_customer'] > 24 else 0)
    score += 2 if row['account_balance'] > 2000000 else (1 if row['account_balance'] > 800000 else 0)
    score += 1 if row['num_monthly_txns'] > 20 else 0
    score += np.random.randint(-1, 2)
    return int(score >= 10)

df['label'] = df.apply(synthetic_label, axis=1)
print(f"Positive rate: {df['label'].mean():.1%} ({df['label'].sum()} / {len(df)})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'], rot=0)
axes[0].set_title('Label Distribution (0=No Convert, 1=Convert)')
axes[0].set_xlabel('')

df.groupby('label')['credit_score'].hist(bins=20, ax=axes[1], alpha=0.6, label=['No Convert','Convert'])
axes[1].set_title('Credit Score by Conversion Label')
axes[1].legend(['No Convert','Convert'])

plt.tight_layout()
plt.show()

## 3. Correlation Analysis

In [ ]:
features = ['monthly_income','account_balance','credit_score','age','num_products',
            'has_salary_account','has_fixed_deposit','months_as_customer',
            'avg_monthly_txn_amount','num_monthly_txns','last_product_purchase_months','education_encoded','label']

corr = df[features].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Model Training — XGBoost + SMOTE

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from imblearn.over_sampling import SMOTE

FEATURE_COLS = ['monthly_income','account_balance','credit_score','age','num_products',
                'has_salary_account','has_fixed_deposit','months_as_customer',
                'avg_monthly_txn_amount','num_monthly_txns','last_product_purchase_months','education_encoded']

X = df[FEATURE_COLS].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
model.fit(X_res, y_res)

y_prob = model.predict_proba(X_test)[:,1]
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}')

## 5. Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()

# Feature Importance
imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
imp.plot(kind='barh', ax=axes[1], color='seagreen')
axes[1].set_title('Feature Importances (XGBoost)')

# Score Distribution
axes[2].hist(y_prob[y_test==0], bins=20, alpha=0.6, label='No Convert', color='coral')
axes[2].hist(y_prob[y_test==1], bins=20, alpha=0.6, label='Convert', color='steelblue')
axes[2].set_xlabel('Propensity Score'); axes[2].set_title('Score Distribution by Label')
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Score the Full Dataset — Top Conversion Candidates

In [ ]:
df['propensity_score'] = model.predict_proba(X)[:,1]
df['tier'] = df['propensity_score'].apply(lambda s: 'High' if s>=0.65 else ('Medium' if s>=0.35 else 'Low'))

top = df[df['has_personal_loan']==False].nlargest(10, 'propensity_score')
top[['name','city','occupation','monthly_income','credit_score','propensity_score','tier']]\
  .style.background_gradient(subset='propensity_score', cmap='Greens').format({'monthly_income':'₹{:,.0f}','propensity_score':'{:.0%}'})

In [ ]:
print('Tier distribution:')
print(df['tier'].value_counts())
print(f"\nTop 10 avg propensity score: {top['propensity_score'].mean():.0%}")
print(f"Cross-sell pool (no existing loan): {(df['has_personal_loan']==False).sum()} customers")